#### 교차검증과 최적의 파라미터 찾기
- 머신러닝을 사용할때 모델의 정확도를 측정하기 위해 반드시 사용해야 하는 방법 = 교차검증
- 딥러닝시에는 데이터의 크기가 크므로 이 방법은 사용할 필요가 없다.


In [1]:
import pandas as pd
wine = pd.read_csv("../Data/wine.csv")
wine.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [2]:
wine.info()

<class 'pandas.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   alcohol  6497 non-null   float64
 1   sugar    6497 non-null   float64
 2   pH       6497 non-null   float64
 3   class    6497 non-null   float64
dtypes: float64(4)
memory usage: 203.2 KB


----
#### Feature와 Target 분리

In [3]:
data = wine[['alcohol','sugar','pH']].to_numpy()
target = wine[['class']].to_numpy()

----
#### 검증 세트 추가
- 19번에서 훈련세트와 테스트세트만 가지고 작업을 하였지만 테스트 세트 작업에 파라미터를 조절하여 정확성을 높히면 실전 사용시 문제가 발생한다.
- 이런 과정을 방지하기 위해 훈련세트, 검증세트, 테스트세트로 구분하여 분석 작업을 한다.

In [4]:
# 전체세트중 훈련세트와 테스트세트를 8:2로 기준으로 분리
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = \
train_test_split(
   data,
   target,
   test_size=0.2,
   random_state=42
   
)

In [5]:
# 훈련세트중 훈련세트와 검증세트
sub_input, val_input, sub_target, val_target = \
   train_test_split(
      train_input,
      train_target,
      test_size=0.2,
      random_state=42
   )

In [6]:
# 훈련세트, 검증세트, 테스트세트의 크기 구하기
print("Train :", sub_input.shape)
print("Valid :", val_input.shape)
print("Test :", test_input.shape)

Train : (4157, 3)
Valid : (1040, 3)
Test : (1300, 3)


In [7]:
# 훈련세트와 검증세트로 결정트리 모델 만들기
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
   min_impurity_decrease=0.0005
)
dt.fit(sub_input, sub_target)

print("Train :", dt.score(sub_input,sub_target))
print("Valid :", dt.score(val_input,val_target))


Train : 0.8946355544864084
Valid : 0.8653846153846154


In [8]:
# Test Set로 최종 확인
print("Test:", dt.score(test_input, test_target))

Test: 0.8638461538461538


----
#### 교차검증(Cross Validation)
- 교차검증이 한 파트를 폴드라고 하며 교차검증의 기본 Fold는 5이다.
- 훈련세트와 검증세트를 바꾸어 가며 정확도를 구하는 방법이다.
- 전체의 대한 정확도는 해당 값들의 평균으로 구한다.

In [9]:
from sklearn.model_selection import cross_validate
scores = cross_validate(dt,train_input,train_target)
scores

{'fit_time': array([0.00745606, 0.00886703, 0.00573087, 0.01190305, 0.00933075]),
 'score_time': array([0.00161815, 0.00311494, 0.00125527, 0.00204396, 0.00196815]),
 'test_score': array([0.86538462, 0.86923077, 0.8825794 , 0.84985563, 0.87102984])}

In [10]:
scores['test_score'].mean()

np.float64(0.8676160509365515)

----
#### Optuna
- 최적화 알고리즘 기반 탐색
- 정밀하고 효율적인 Hypler Parameter 탐색

In [11]:
#! pip install optuna

In [12]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
iris = load_iris()

train_input, test_input, train_target, test_target = \
   train_test_split(
      iris.data,
      iris.target,
      test_size=0.2,
      random_state=42,
      stratify=iris.target
   )

In [14]:
def find_param(trial):
   # 탐색할 하이퍼파라미터 정의
   n_estimators = trial.suggest_int('n_estimator', 50, 200)
   max_depth = trial.suggest_int('max_depth', 2, 20)
   min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
   min_samples_leaf = trial.suggest_int('min_samples_leaf',1,5)
   bootstrap = trial.suggest_categorical('bootstrap', [True, False])

   clf = RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      min_samples_split=min_samples_split,
      min_samples_leaf=min_samples_leaf,
      bootstrap=bootstrap,
      random_state=42
   )

   return cross_val_score(clf,train_input,train_target,cv=5).mean()

In [15]:
study = optuna.create_study(direction='maximize')
study.optimize(find_param, n_trials=50)

print("최적의 파라미터 :", study.best_params)

[I 2026-07-03 14:14:08,339] A new study created in memory with name: no-name-8dc14954-4ef0-40b3-9cc1-11374b4f4c9c
[I 2026-07-03 14:14:09,064] Trial 0 finished with value: 0.9416666666666668 and parameters: {'n_estimator': 57, 'max_depth': 18, 'min_samples_split': 5, 'min_samples_leaf': 4, 'bootstrap': True}. Best is trial 0 with value: 0.9416666666666668.
[I 2026-07-03 14:14:10,699] Trial 1 finished with value: 0.95 and parameters: {'n_estimator': 156, 'max_depth': 7, 'min_samples_split': 10, 'min_samples_leaf': 2, 'bootstrap': True}. Best is trial 1 with value: 0.95.
[I 2026-07-03 14:14:12,167] Trial 2 finished with value: 0.9583333333333334 and parameters: {'n_estimator': 186, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 4, 'bootstrap': False}. Best is trial 2 with value: 0.9583333333333334.
[I 2026-07-03 14:14:13,734] Trial 3 finished with value: 0.95 and parameters: {'n_estimator': 129, 'max_depth': 18, 'min_samples_split': 2, 'min_samples_leaf': 5, 'bootstrap': Tru

최적의 파라미터 : {'n_estimator': 186, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 4, 'bootstrap': False}


In [19]:
# 최종모델 Hyper Parameter 조정하가
# 랜덤포레스트 분류기

rf = RandomForestClassifier(
      n_estimators=186,
      max_depth=15,
      min_samples_split=5,
      min_samples_leaf=4,
      bootstrap=False,
      random_state=42
   )

In [20]:
rf.fit(train_input,train_target)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",186
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",4
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",False
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the rig

In [21]:
print("Train :", rf.score(train_input,train_target))
print("Test :", rf.score(test_input,test_target))

Train : 0.9833333333333333
Test : 0.9666666666666667
